# Animation Studio \u2014 Character LoRA Training (Phase 1c)

Trains character-consistency LoRAs with kohya sd-scripts `flux_train_network.py` on a Colab GPU runtime, then benchmarks each against the studio identity scorer and syncs the artifacts back to the AnimationStudio GitHub repo.

## What this notebook does (run top-to-bottom, edit only the **Settings** cell)

1. Settings (runtime parameters + token placeholders + batch-scope `CHARACTERS`)
2. GPU check \u2014 fails fast with guidance on no-GPU / under-powered runtimes
3. Clones the AnimationStudio repo, installs the studio, pins kohya `sd-scripts` to a fixed upstream commit (supply-chain pin)
4. Downloads the four FLUX.1-dev model files (gated \u2192 requires `HF_TOKEN`)
5. Defines `build_dataset(character_id)` (per-character dataset curation)
6. Defines `train_lora(character_id, character_title, dataset_config)` (`accelerate launch flux_train_network.py`, VRAM profiles for A100 / T4 / 12 GB)
7. Defines `benchmark_lora(...)` (diffusers Flux sample generation + `LoRABenchmark` + `IdentityScorerProvider(light=False)`)
8. Defines `register_promote(...)` (register the version; promote when the benchmark gate passes)
9. Defines `sync_artifacts(...)` (commit `*.safetensors` + benchmark report + registry to GitHub)
10. **Batch driver**: runs the full pipeline for every selected character \u2014 skipping characters whose registry entry is already promoted and benchmark-passing

## Prerequisites

- **GPU Colab runtime**: T4 GPU (free tier) or A100 GPU. CPU-only runtimes fail fast in cell 2.
- **Hugging Face token** (`hf_...`) for the gated **FLUX.1-dev** model \u2014 accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev, then paste the token in the Settings cell.
- **GitHub fine-grained PAT** with *Contents: Read and write* on the AnimationStudio repo (only needed when `SYNC_TO_GITHUB = True`, cell 9).

## Security note

This notebook commits with **empty token placeholders**. `HF_TOKEN` and `GITHUB_TOKEN` are runtime session values only \u2014 they are never written back into the notebook and never committed to git.

## Time expectations

Hours-scale **per character** on free T4 (\u22485 s/it @512 px with the fp8 profile). Watch Colab session limits; a failed GPU session means re-running from the top (already-synced artifacts are skipped idempotently via the registry gate).

In [ ]:
#@title 1. Settings

import os
import pathlib
import subprocess
import sys

# --- Repo -------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "colab-gpu"  #@param ["colab-gpu"]

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
SD_SCRIPTS_DIR = f"{WORK}/sd-scripts"

# --- Training run -----------------------------------------------------------
CHARACTER_ID = "lily-bunny"  #@param {type:"string"}
CHARACTER_TITLE = "Lily Bunny"  # Universe directory display name (Universe/Characters/<title>)
VRAM_PROFILE = "t4-16g"  #@param ["a100-24g", "t4-16g", "t4-12gb-swap16"]

# Training artifacts live inside the repo clone so the sync cell can commit them.
TRAINING_ROOT = f"{REPO}/training"
OUTPUT_DIR = f"{REPO}/Universe/Characters/{CHARACTER_TITLE}/lora"
SAMPLES_DIR = f"{TRAINING_ROOT}/samples"
MODEL_DIR = f"{WORK}/models"
REGISTRY_PATH = f"{TRAINING_ROOT}/lora_registry.json"

# --- Batch training scope (N-07) --------------------------------------------
# Pick ONE character for a send-ahead single run, a comma-separated list of
# names, or 'all' for the full 39-character universe.  The Cell 10 driver runs
# the whole build -> train -> benchmark -> register/promote -> sync pipeline
# per selected character.  Free-T4 note: each character is hours-scale, so
# prefer one character at a time on the free tier.
CHARACTERS = "Lily Bunny"  #@param ["all", "Ben Bear", "Charlie Fox", "Daisy Duck", "Lily Bunny", "Baby Bunny", "Daddy Bunny", "Grandma Bunny", "Grandpa Bunny", "Mommy Bunny", "Cat", "Chicken", "Cow", "Dog", "Elephant", "Horse", "Monkey", "Mouse", "Pig", "Sheep", "Chef Pig", "Construction Worker Beaver", "Doctor Panda", "Farmer Goat", "Firefighter Dalmatian", "Librarian Hedgehog", "Mail Carrier Turtle", "Musician Parrot", "Police Officer Beaver", "Teacher Owl", "Alien", "Cloud", "Friendly Dinosaur (Brontosaurus)", "Friendly Dragon", "Moon", "Rainbow", "Robot", "Stars", "Sun", "Unicorn"]

# Cap the number of characters processed per driver run (0 = no cap).  Set to 1
# to trial-run exactly one character before committing hours of GPU time.
MAX_CHARACTERS_PER_RUN = 0  #@param {type:"integer"}

# Skip characters whose registry entry is already promoted with a passing
# benchmark gate (idempotent re-runs, e.g. after a Colab session reset).
SKIP_ALREADY_TRAINED = True  #@param {type:"boolean"}

# Send-ahead: comma-separated character IDs that override CHARACTERS for a
# partial run (e.g. "lily-bunny,ben-bear").  Empty = use CHARACTERS.
SEND_AHEAD_IDS = ""  #@param {type:"string"}

# --- GitHub sync ------------------------------------------------------------
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# --- Secrets (runtime-only; never commit real values) -----------------------
GITHUB_TOKEN = ""  #@param {type:"string"}
HF_TOKEN = ""      #@param {type:"string"}

In [ ]:
#@title 2. GPU check (fail fast)

import shutil

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU detected. This notebook trains FLUX.1-dev LoRAs and needs a "
        "GPU Colab runtime: Runtime > Change runtime type > T4 GPU (free) or A100 GPU, "
        "then re-run from cell 1."
    )
subprocess.run(["nvidia-smi"], check=True)  # device summary

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "torch reports CUDA unavailable. Verify the runtime accelerator is set to a "
        "GPU (Runtime > Change runtime type), then re-run from cell 1."
    )

device = torch.cuda.get_device_name(0)
COMPUTE = torch.cuda.get_device_capability()
COMPUTE_MAJOR = COMPUTE[0]
print(
    f"Device: {device} | CUDA {torch.version.cuda} | torch {torch.__version__} "
    f"| compute capability {COMPUTE[0]}.{COMPUTE[1]}"
)
if COMPUTE_MAJOR < 8:
    print("NOTE: Turing/older GPUs have no native bf16; fp16 raises NaN on Flux.")
    print("The training cell forces the fp8_base path (emulated, ~5 s/it @512 px) \u2014 expect hours-scale runs.")

In [ ]:
#@title 3. Clone the repo, install the studio, pin kohya sd-scripts

def run(argv):
    print("+ " + " ".join(str(a) for a in argv))
    subprocess.run([str(a) for a in argv], check=True)

# --- AnimationStudio --------------------------------------------------------
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
run(["git", "-C", REPO, "checkout", BRANCH])
run(["git", "-C", REPO, "pull", "origin", BRANCH])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

# --- kohya sd-scripts (pinned to a fixed upstream commit for reproducibility) ----
SD_SCRIPTS_URL = "https://github.com/kohya-ss/sd-scripts.git"
SD_SCRIPTS_PIN = "37a1cbbc5725ed2a3575506e7bd2001c9908ac92"  # main @ 2026-07-23
if not os.path.isdir(SD_SCRIPTS_DIR):
    run(["git", "clone", SD_SCRIPTS_URL, SD_SCRIPTS_DIR])
run(["git", "-C", SD_SCRIPTS_DIR, "checkout", SD_SCRIPTS_PIN])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{SD_SCRIPTS_DIR}/requirements.txt"])

In [ ]:
#@title 4. Download the four FLUX.1-dev model files

# Four files required by flux_train_network.py (see sd-scripts FLUX LoRA guide):
#   flux1-dev.safetensors + ae.safetensors  <- black-forest-labs/FLUX.1-dev (gated)
#   clip_l.safetensors + t5xxl_fp16.safetensors <- comfyanonymous/flux_text_encoders
pathlib.Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

from huggingface_hub import hf_hub_download

def _download(repo_id, filename):
    try:
        return hf_hub_download(
            repo_id, filename, local_dir=MODEL_DIR, token=HF_TOKEN or None
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to download {filename} from {repo_id}. Gated FLUX.1-dev files "
            f"require an HF token that accepted the model license "
            f"(https://huggingface.co/settings/tokens + "
            f"https://huggingface.co/black-forest-labs/FLUX.1-dev). Cause: {exc}"
        ) from exc

MODEL_FLUX = _download("black-forest-labs/FLUX.1-dev", "flux1-dev.safetensors")
MODEL_AE = _download("black-forest-labs/FLUX.1-dev", "ae.safetensors")
MODEL_CLIP_L = _download("comfyanonymous/flux_text_encoders", "clip_l.safetensors")
MODEL_T5XXL = _download("comfyanonymous/flux_text_encoders", "t5xxl_fp16.safetensors")

# N-10: known-good sizes (floors) for each file in the FLUX.1-dev bundle.
EXPECTED_BYTES = {
    "flux1-dev.safetensors": 33956436064,   # FLUX.1-dev fp16 (gated)
    "ae.safetensors": 334867980,
    "clip_l.safetensors": 257964776,
    "t5xxl_fp16.safetensors": 9528282880,
}

# N-08: the fp16 Flux dev + encoder/VAE bundle needs ~45 GB of VM disk.
import shutil
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"Free disk on {WORK}: {free_gb:.1f} GB (FLUX fp16 bundle needs ~45 GB)")
if free_gb < 50:
    raise RuntimeError(
        f"Only {free_gb:.1f} GB free on {WORK} \u2014 the FLUX.1-dev fp16 bundle "
        "needs ~45 GB. Stop other Colab runtimes or free disk space."
    )

for p in (MODEL_FLUX, MODEL_AE, MODEL_CLIP_L, MODEL_T5XXL):
    size_b = os.path.getsize(p)
    expected = EXPECTED_BYTES.get(os.path.basename(p))
    print(f"{os.path.basename(p)}: {size_b / 1e9:.2f} GB")
    if size_b == 0:
        raise RuntimeError(f"Downloaded file is empty: {p}")
    if expected is not None and size_b < expected * 0.99:
        raise RuntimeError(
            f"{os.path.basename(p)} looks truncated: {size_b / 1e9:.2f} GB "
            f"< expected {expected / 1e9:.2f} GB \u2014 re-run this cell "
            "(hf_hub_download resumes) or delete the file and restart."
        )

In [ ]:
#@title 5. Build the training dataset from curated assets (per character)

def build_dataset(character_id, output_root):
    """build-dataset pulls identity-locked curated assets (states:
    approved,production) from catalog.db and writes dataset_config.toml,
    train/ + val/ subsets and baselines/ reference images under output_root.

    Requires at least 20 curated images per character; below that the CLI exits
    non-zero printing per-state counts -- promote more assets in the Review UI
    first.  Returns the dataset_config.toml path for the character.
    """
    try:
        run([
            sys.executable, "scripts/train_lora.py", "build-dataset",
            "--db", "catalog.db",
            "--character-id", character_id,
            "--output-root", output_root,
            "--min-images", "20",
            "--max-images", "40",
        ])
    except subprocess.CalledProcessError as exc:
        print(
            "\nDataset build failed (see output above). If the curated set is under the "
            "20-image minimum, promote additional assets to approved/production via the "
            "Review UI, then re-run the driver cell (Cell 10)."
        )
        raise

    dataset_config = f"{output_root}/dataset_config.toml"
    print("\nDataset config:", dataset_config)
    return dataset_config

In [ ]:
#@title 6. Train the LoRA (accelerate + flux_train_network.py, per character)

def train_lora(character_id, character_title, dataset_config):
    """Train one character's LoRA at the registry-recommended next version.

    Returns (next_version, version_str, lora_name, lora_path).
    """
    # Next recommended version for this character (v0.1 on a fresh registry).
    from src.training_engine.version_store import load_registry

    registry = load_registry(pathlib.Path(REGISTRY_PATH))
    next_version = registry.recommend_next(character_id, "minor")
    version_str = str(next_version)
    print(f"Training {character_id} -> {version_str} (registry-recommended next version)")

    # VRAM profiles (cited in 01c-RESEARCH.md):
    #   a100-24g       : batch 2, native bf16, no fp8, no swap
    #   t4-16g         : batch 1 + --fp8_base + --blocks_to_swap 8  (Turing bf16 emulated)
    #   t4-12gb-swap16 : batch 1 + --fp8_base + --blocks_to_swap 16 + AdamW8bit
    VRAM_PROFILES = {
        "a100-24g":       {"batch_size": 2, "fp8_base": False, "blocks_to_swap": 0},
        "t4-16g":         {"batch_size": 1, "fp8_base": True,  "blocks_to_swap": 8},
        "t4-12gb-swap16": {"batch_size": 1, "fp8_base": True,  "blocks_to_swap": 16},
    }
    if VRAM_PROFILE not in VRAM_PROFILES:
        raise ValueError(f"Unknown VRAM_PROFILE: {VRAM_PROFILE!r} (choose from {list(VRAM_PROFILES)})")
    profile = VRAM_PROFILES[VRAM_PROFILE]

    # Pitfall 6: never force fp8_base on compute < 8 even for the a100 profile.
    use_fp8 = profile["fp8_base"] or COMPUTE_MAJOR < 8
    if use_fp8 and not profile["fp8_base"]:
        print("NOTE: compute capability < 8 -> forcing --fp8_base despite selected profile.")

    lora_name = f"{character_id}_{version_str}"
    output_name = lora_name  # -> Universe/Characters/<title>/lora/<id>_<version>.safetensors
    output_dir = f"{REPO}/Universe/Characters/{character_title}/lora"

    cmd = [
        "accelerate", "launch", "--num_cpu_threads_per_process", "1",
        f"{SD_SCRIPTS_DIR}/flux_train_network.py",
        "--pretrained_model_name_or_path", MODEL_FLUX,
        "--clip_l", MODEL_CLIP_L,
        "--t5xxl", MODEL_T5XXL,
        "--ae", MODEL_AE,
        "--dataset_config", dataset_config,
        "--output_dir", output_dir,
        "--output_name", output_name,
        "--save_model_as", "safetensors",
        "--network_module", "networks.lora_flux",
        "--network_dim", "32",
        "--network_alpha", "32",
        "--learning_rate", "1e-4",
        "--optimizer_type", "AdamW8bit",
        "--lr_scheduler", "constant",
        "--max_train_epochs", "10",
        "--mixed_precision", "bf16",
        "--save_precision", "bf16",
        "--seed", "42",
        "--gradient_checkpointing",
        "--sdpa",
        "--train_batch_size", str(profile["batch_size"]),
        "--cache_latents",
        "--cache_latents_to_disk",
        "--cache_text_encoder_outputs",
        "--cache_text_encoder_outputs_to_disk",
        "--guidance_scale", "1.0",
        "--timestep_sampling", "flux_shift",
        "--model_prediction_type", "raw",
        "--max_data_loader_n_workers", "2",
    ]
    if use_fp8:
        cmd += ["--fp8_base"]
    if profile["blocks_to_swap"] > 0:
        cmd += ["--blocks_to_swap", str(profile["blocks_to_swap"])]

    # Long-running step: hours-scale on free T4. Do not interrupt; watch session limits.
    run(cmd)

    lora_path = pathlib.Path(output_dir) / f"{lora_name}.safetensors"
    if not lora_path.exists():
        raise RuntimeError(f"Training finished but {lora_path} is missing -- check the accelerate log above.")
    print(f"\nSaved: {lora_path}")
    return next_version, version_str, lora_name, lora_path

In [ ]:
#@title 7. Generate samples and benchmark the LoRA (per character)

def benchmark_lora(character_id, character_title, version_str, lora_name, lora_path, char_root):
    """Generate N=8 samples with a diffusers Flux pipeline using the trained
    LoRA, then score against the identity scorer baseline.

    Returns (result, report_path).
    """
    # N=8 samples total, saved under {char_root}/samples: the 5
    # BenchmarkConfig.test_prompts (single source of truth) plus 3 extra
    # visual-review prompts. All 8 feed the benchmark below.
    from diffusers import FluxPipeline
    from src.training_engine.benchmark import BenchmarkConfig

    output_dir = f"{REPO}/Universe/Characters/{character_title}/lora"

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN or None,
    )
    pipe.load_lora_weights(output_dir, weight_name=f"{lora_name}.safetensors")
    pipe.to("cuda")

    samples_dir = pathlib.Path(f"{char_root}/samples")
    samples_dir.mkdir(parents=True, exist_ok=True)

    _bench_prompts = [p for p, _ in BenchmarkConfig().test_prompts]
    _review_prompts = [
        "smiling, looking at camera, soft morning light",
        "waving, playful pose, outdoor park background",
        "curious expression, head tilt, cozy indoor scene",
    ]
    sample_prompts = _bench_prompts + _review_prompts
    sample_images = []
    for i, prompt in enumerate(sample_prompts):
        gen = torch.Generator("cuda").manual_seed(42 + i)
        image = pipe(
            prompt=f"{character_title}, {prompt}",
            num_inference_steps=28,
            guidance_scale=3.5,
            generator=gen,
        ).images[0]
        out_path = samples_dir / f"sample-{i + 1:02d}.png"
        image.save(out_path)
        sample_images.append(out_path)
        print(f"sample-{i + 1:02d}.png <- {prompt!r}")

    # Benchmark against the identity scorer baseline (full plugin stack on Colab).
    from src.training_engine.benchmark import LoRABenchmark
    from src.training_engine.scorer_adapter import IdentityScorerProvider

    benchmark = LoRABenchmark(
        scorer_provider=IdentityScorerProvider(light=False),  # full DINOv2/CLIP plugins
        config=BenchmarkConfig(baseline_dir=pathlib.Path(f"{char_root}/baselines")),
    )
    result = benchmark.evaluate(
        lora_path=lora_path,
        character_id=character_id,
        test_images=sample_images,
    )
    report = benchmark.report(result)
    print(report)
    print(f"Gate: composite >= 0.90 AND weight coverage 100% -> passed={result.passed}")

    report_path = pathlib.Path(output_dir) / f"benchmark_report_{version_str}.md"
    report_path.write_text(report, encoding="utf-8")
    print(f"Report saved: {report_path}")
    return result, report_path

In [ ]:
#@title 8. Register and promote the version (per character)

def register_promote(character_id, next_version, version_str, lora_path, result,
                     dataset_config, output_dir):
    """Register always; promote only when the benchmark gate passed (composite
    >= 0.90 with full weight coverage, per CHAR-07 / Phase 1c success criteria).

    Real training completion on Colab -- dry-run registration semantics do NOT
    apply here.
    """
    training_config_used = {
        "base_model": "black-forest-labs/FLUX.1-dev",
        "network_module": "networks.lora_flux",
        "network_dim": 32,
        "network_alpha": 32,
        "learning_rate": 1e-4,
        "optimizer_type": "AdamW8bit",
        "lr_scheduler": "constant",
        "max_train_epochs": 10,
        "mixed_precision": "bf16",
        "resolution": 1024,  # must match cache in dataset_config.toml
        "seed": 42,
        "vram_profile": VRAM_PROFILE,
        "dataset_config": dataset_config,
        "output_dir": output_dir,
    }

    benchmark_scores = {dim.name: dim.score for dim in result.dimensions}
    benchmark_scores["composite"] = result.composite_score
    benchmark_scores["passed"] = result.passed

    from src.training_engine.version_store import load_registry
    registry = load_registry(pathlib.Path(REGISTRY_PATH))
    registry.register(
        character_id=character_id,
        version=next_version,
        file_path=str(lora_path),
        training_config=training_config_used,
        benchmark_scores=benchmark_scores,
    )

    if result.passed:
        registry.promote(character_id, next_version)
        print(f"\u2705 Promoted {character_id} {version_str} to production (benchmark gate passed).")
    else:
        print(
            f"\u274c Benchmark gate FAILED for {version_str} (composite={result.composite_score:.2%}). "
            f"Version is registered but NOT promoted -- retrain with more/better data."
        )
    print("Registry:", REGISTRY_PATH)

In [ ]:
#@title 9. Sync artifacts to GitHub (or manual download, per character)

def sync_artifacts(character_id, character_title, version_str, lora_path, report_path):
    """Commit one character's LoRA + benchmark report + registry to GitHub."""
    from datetime import datetime

    if SYNC_TO_GITHUB:
        if not GITHUB_TOKEN:
            raise RuntimeError(
                "SYNC_TO_GITHUB is on but GITHUB_TOKEN is empty. Create a fine-grained PAT "
                "with Contents: Read and write on AnimationStudio, paste it into Settings, "
                "or set SYNC_TO_GITHUB = False for manual download."
            )
        sys.path.insert(0, f"{REPO}/colab")
        from git_sync import _basic_auth_header

        run(["git", "config", "user.name", GIT_NAME])
        run(["git", "config", "user.email", GIT_EMAIL])

        # Extended add vs Phase 4 pattern: LoRA safetensors + benchmark report
        # (both live in Universe/Characters/<title>/lora/) + registry + dataset manifest.
        run(["git", "add", f"Universe/Characters/{character_title}/lora"])
        for rel in ["training/lora_registry.json", f"training/{character_id}/dataset_config.toml"]:
            if pathlib.Path(rel).exists():
                run(["git", "add", rel])

        staged = subprocess.run(
            ["git", "diff", "--cached", "--name-only"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if not staged:
            print("Nothing new to commit -- artifacts already synced on a prior run (idempotent).")
        else:
            stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            run(["git", "commit", "-m", f"feat(training): {character_id} {version_str} LoRA + benchmark ({stamp})"])
            run(["git", "-c", f"http.extraheader=Authorization: {_basic_auth_header(GITHUB_TOKEN)}", "push", "origin", BRANCH])
            print("Pushed to GitHub.")
    else:
        from google.colab import files
        files.download(str(lora_path))
        files.download(str(report_path))
        print("Manual download started for LoRA + benchmark report.")

In [ ]:
#@title 10. Batch driver: train every selected character (N-07)

# One run drives the full pipeline for every character in CHARACTERS (Cell 1),
# calling the per-character functions from Cells 5-9:
#   build_dataset -> train_lora -> benchmark_lora -> register_promote -> sync_artifacts
#
# Characters already promoted with a benchmark-passing version in
# training/lora_registry.json are skipped (SKIP_ALREADY_TRAINED).  Send-ahead:
# pick one name in CHARACTERS, or set SEND_AHEAD_IDS to a comma-separated list
# of character IDs for a partial run before committing hours of GPU time.

import sys
sys.path.insert(0, REPO)

from src.universe.catalog import discover_characters

seeds = discover_characters(f"{REPO}/Universe")
if not seeds:
    raise SystemExit(f"No characters found under {REPO}/Universe - check the clone.")

by_slug = {s.slug: s for s in seeds}
by_name = {s.name: s for s in seeds}

send_ahead = [x.strip().lower() for x in SEND_AHEAD_IDS.split(",") if x.strip()]
if send_ahead:
    missing = [x for x in send_ahead if x not in by_slug]
    if missing:
        raise SystemExit(f"Unknown SEND_AHEAD_IDS character id(s): {missing}")
    matched = [by_slug[x] for x in send_ahead]
else:
    picks = [x.strip() for x in CHARACTERS.split(",") if x.strip()]
    if "all" in {x.lower() for x in picks}:
        matched = list(seeds)
    else:
        matched = []
        for name in picks:
            if name not in by_name:
                raise SystemExit(f"Unknown CHARACTERS name: {name!r}. Available: {sorted(by_name)}")
            matched.append(by_name[name])

if MAX_CHARACTERS_PER_RUN and MAX_CHARACTERS_PER_RUN > 0:
    matched = matched[:MAX_CHARACTERS_PER_RUN]


def _already_trained(seed) -> bool:
    """True when the registry holds a promoted, benchmark-passing version."""
    if not SKIP_ALREADY_TRAINED:
        return False
    from src.training_engine.version_store import load_registry
    registry = load_registry(pathlib.Path(REGISTRY_PATH))
    promoted = registry.get_promoted(seed.slug)
    return bool(
        promoted is not None
        and promoted.benchmark_scores
        and promoted.benchmark_scores.get("passed")
    )


print("=" * 72)
print(f"Training scope: {len(matched)} character(s)")
for s in matched:
    print(f"  - {s.name} ({s.slug})")
print("=" * 72)

for idx, seed in enumerate(matched, 1):
    character_id = seed.slug
    character_title = seed.name
    if _already_trained(seed):
        print(f"[{idx}/{len(matched)}] SKIP {character_title}: already trained and passing in {REGISTRY_PATH}")
        continue

    print(f"\n[{idx}/{len(matched)}] {character_title} ({character_id}) ==============================")
    char_root = f"{TRAINING_ROOT}/{character_id}"
    pathlib.Path(char_root).mkdir(parents=True, exist_ok=True)

    print("\n--- (a) build-dataset ---")
    dataset_config = build_dataset(character_id, char_root)

    print("\n--- (b) train ---")
    next_version, version_str, lora_name, lora_path = train_lora(
        character_id, character_title, dataset_config
    )

    print("\n--- (c) generate samples + benchmark ---")
    result, report_path = benchmark_lora(
        character_id, character_title, version_str, lora_name, lora_path, char_root
    )

    print("\n--- (d) register + promote ---")
    output_dir = f"{REPO}/Universe/Characters/{character_title}/lora"
    register_promote(
        character_id, next_version, version_str, lora_path, result,
        dataset_config, output_dir,
    )

    print("\n--- (e) sync ---")
    sync_artifacts(character_id, character_title, version_str, lora_path, report_path)

print("\nBatch training complete.")

## Next steps

- **Verify the commits**: check GitHub for `Universe/Characters/<title>/lora/` (safetensors + benchmark report) and `training/lora_registry.json` for each trained character.
- **Review remaining assets**: promote more curated assets through the Review UI (`nursery review` / HTTP UI) if a curated set was under 20 images \u2014 then re-run the driver (Cell 10).
- **Use the LoRAs downstream**: load the promoted `*.safetensors` in image generation via `scripts/generate_phase1_library.py --backend comfyui` (Phase 1b / `src/image_generation` + ComfyUI workflow `CharacterLock`) or diffusers `load_lora_weights`, always with the identity-locked prompt system; Phase 9 animation consumes the same generated assets.
- **Other characters in one run**: set `CHARACTERS` in cell 1 \u2014 one name for a send-ahead single run, a comma-separated list for a batch, or `all` for the full 39-character universe. Characters whose registry entry is already promoted and benchmark-passing are skipped automatically (`SKIP_ALREADY_TRAINED`).
- **Partial / send-ahead runs**: set `SEND_AHEAD_IDS` (e.g. `lily-bunny,ben-bear`) to train a specific subset before committing hours of GPU time, or `MAX_CHARACTERS_PER_RUN = 1` for exactly one character.
- **Benchmark evidence**: this notebook\u2019s benchmark reports + registry JSON are the recorded evidence for the deferred-human verification of the LOv1 production LoRA criterion (CHAR-07).
- **Session limits**: on free T4 expect hours-scale; if the session dies mid-training, re-run from the top \u2014 dataset build is cheap, sync is idempotent, and the registry gate skips already-finished characters.